<a href="https://colab.research.google.com/github/ymuto0302/RW2025/blob/main/MLP_pytorch_basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch を用いた MLP の実装
(注) PyTorch におけるモデル定義に注目してほしいため，Dataset と Dataloader を使用しない設計としている。Dataset と Dataloader を使用する設計は以下の参考資料を参照してください。
https://github.com/ymuto0302/RW2025/blob/main/MLP_pytorch_dataset_dataloader.ipynb

## 1. 手書きで書いた数字分類の練習

### 1.1. ライブラリのインポート
PyTorch を利用する場合，`torch`，ネットワーク定義と損失関数定義に用いる `torch.nn`, 最適化アルゴリズムを定義する `torch.optim` をインポートする。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt
import torch.optim as optim
import torch.nn as nn
import numpy as np
import torch

### 1.2. データ生成
PyTorch ではモデルに与えるデータ形式がテンソルであるため，その変換部を実装した。

（注意）MLP ではデータの標準化が推奨されている。本来ならば，訓練データで学習した scaler をテストデータへ適用すべきだが，（ちょっと手を抜いて）訓練・テストへの分割前に標準化を行っている。

In [ ]:
# データの準備: 手書き数字データセット MNIST を用いる
def prepare_data(device='cpu'):

    # 手書き数字データセットの読み込み
    digits = load_digits()
    X, y = digits.data, digits.target

    # データの標準化
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # NumPy配列をPyTorchテンソルに変換
    X_tensor = torch.FloatTensor(X).to(device)
    y_tensor = torch.LongTensor(y).to(device)

    # 訓練用とテスト用に分割
    n_train = int(0.8 * len(X)) # 訓練データのサイズ
    X_train = X_tensor[:n_train]
    y_train = y_tensor[:n_train]
    X_test = X_tensor[n_train:]
    y_test = y_tensor[n_train:]

    return X_train, y_train, X_test, y_test

### 1.3. モデルの定義
MLPは2層のモデルであるため，モデルは 2つの`Linear()`と`ReLU`の活性化関数で構成される。

In [ ]:
# MLPモデルの定義
class MLP(nn.Module):

    def __init__(self, input_size, output_size):
        super(MLP, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32,  16),
            nn.ReLU(),
            nn.Linear(16, output_size))

    def forward(self, x):
        return self.network(x)

### 1.4. モデルの学習関数の定義

In [ ]:
# モデルを学習するための関数
def train_model(model, X_train, y_train, X_test, y_test, epochs=100, batch_size=32, solver="adam"):
    # 損失関数とオプティマイザー
    criterion = nn.CrossEntropyLoss()
    if solver == "adam":
        optimizer = optim.Adam(model.parameters())
    if solver == "sgd":
        optimizer = optim.SGD(model.parameters())


    # 学習履歴を保存
    train_losses = []
    train_accuracies = []
    test_accuracies = []

    # データサイズとバッチ数
    n_samples = len(X_train) # 訓練サンプル数
    n_batches = (n_samples + batch_size - 1) // batch_size

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        # ミニバッチ学習      （注)手動でバッチを作成する
        for i in range(n_batches):
            start_idx = i * batch_size
            end_idx = min((i + 1) * batch_size, n_samples)

            # バッチデータの取得
            batch_X = X_train[start_idx:end_idx]
            batch_y = y_train[start_idx:end_idx]

            # 勾配の初期化
            optimizer.zero_grad()

            # 順伝播
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)

            # 逆伝播
            loss.backward()
            optimizer.step()

            # 損失の累積
            epoch_loss += loss.item()

        # エポックごとの平均損失を記録
        avg_loss = epoch_loss / n_batches
        train_losses.append(avg_loss)

        # テストデータでの精度評価
        model.eval()
        with torch.no_grad():
            # 予測
            test_outputs = model(X_test)
            train_outputs = model(X_train)
            # 確率最大のクラスを取り出す
            _, test_predicted = torch.max(test_outputs.data, 1)
            _, train_predicted = torch.max(train_outputs.data, 1)
            # 正解率を求める
            test_accuracy = (test_predicted == y_test).float().mean().item()
            train_accuracy = (train_predicted == y_train).float().mean().item()
            # 正解率の累積
            test_accuracies.append(test_accuracy)
            train_accuracies.append(train_accuracy)

        # 進捗表示
        if (epoch + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}, Train Acc: {train_accuracy:.4f}, Test Acc: {test_accuracy:.4f}')

    return train_losses, test_accuracies, train_accuracies

### 1.5. データセットの前処理、学習

In [ ]:
# デバイス設定
# GPU が利用可能場合，GPU を利用する
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用デバイス: {device}")

# データの準備
print("データを準備中...")
X_train, y_train, X_test, y_test = prepare_data(device)
print(f"訓練データ: {X_train.shape}, テストデータ: {X_test.shape}")

# モデルの作成
input_size = X_train.shape[1]  # 特徴量数
output_size = 10                # クラス数

model = MLP(input_size, output_size)
model = model.to(device)
print(f"\nモデル構造:\n{model}")

# モデルの学習
print("\nモデルを学習中...")
train_losses, test_accuracies, train_accuracies = train_model(
    model, X_train, y_train, X_test, y_test,
    epochs=200, batch_size=32
)

# 最終的な性能評価
model.eval()
with torch.no_grad():
    # 訓練データでの性能
    train_outputs = model(X_train)
    _, train_predicted = torch.max(train_outputs.data, 1)
    train_accuracy = (train_predicted == y_train).float().mean().item()

    # テストデータでの性能
    test_outputs = model(X_test)
    _, test_predicted = torch.max(test_outputs.data, 1)
    test_accuracy = (test_predicted == y_test).float().mean().item()

print(f"\n最終結果:")
print(f"訓練データに対する正解率: {train_accuracy:.4f}")
print(f"テストデータに対する正解率: {test_accuracy:.4f}")

### 1.6 学習履歴の可視化

In [ ]:
# 学習履歴の可視化
def plot_training_history(train_losses, train_accuracies, test_accuracies=None):
    """学習履歴をプロット"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(train_losses, label='Train Loss')
    ax1.set_title('Training Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.grid(True)

    ### ax2.plot(train_accuracies, label='Train Accuracy', marker='o', markersize=3)
    ax2.plot(train_accuracies, label='Train Accuracy')
    if test_accuracies and len(test_accuracies) > 0:
        ### ax2.plot(test_accuracies, label='Test Accuracy', marker='s', markersize=3)
        ax2.plot(test_accuracies, label='Test Accuracy')
    ax2.set_title('Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

# 学習履歴の可視化
print("\n=== 学習履歴 ===")
plot_training_history(train_losses, test_accuracies, train_accuracies)

## 2. 10クラス2次元分類の演習

### 2.1. データセットの生成、標準化、可視化

In [ ]:
from sklearn.datasets import make_gaussian_quantiles

# 10クラス2次元Gaussian Quantilesデータセットの生成
n_classes = 10
n_features = 2
X, y = make_gaussian_quantiles(n_samples=1000, n_features=n_features, n_classes=n_classes, random_state=42)

print(f"入力データの特徴量: {n_features}, 出力データのラベル: {n_classes}")

# データの分割…






# 標準化…






# テンソル化（同じ変数名に書き換える）
X_train = torch.FloatTensor(X_train).to(device)
y_train = torch.LongTensor(y_train).to(device)
X_test = torch.FloatTensor(X_test).to(device)
y_test = torch.LongTensor(y_test).to(device)

# 生成したデータセットの可視化（おまけ）
plt.figure(figsize=(7, 6))
scatter = plt.scatter(
    X[:, 0],
    X[:, 1],
    c=y,
    cmap="tab10",
    s=25,
    alpha=0.8,
    edgecolors="k",
    linewidths=0.2
)
plt.title("10-class 2D Gaussian Quantiles Dataset")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.colorbar(scatter, label="Class")
plt.grid(True, alpha=0.3)
plt.show()


### 2.2. データの前処理と基本的なMLPモデルの実装

- データを訓練セットとテストセットに分割してください。
- PyTorchの `linear()` レイヤを用いて基本的なMLPモデル（隠れ層のサイズ: (100, 50)）を実装し，訓練してください。
- モデルの性能を評価し，正解率と混同行列を表示してください。

In [ ]:
# MLPモデルの定義
class MultiLayerPerceptron(nn.Module):
    
    def __init__(self, input_size, num_classes):
        # PyTorchの乱数を固定化する
        torch.manual_seed(42)
        super(MultiLayerPerceptron, self).__init__()
        # レイヤを順次的に接続する
        self.network = nn.Sequential(
		    # 入力層
            nn.Linear(input_size, 〇〇〇),
            # 活性化関数
            nn.ReLU(),
            # 隠れ層
            nn.Linear(〇〇〇, △△),
            # 活性化関数
            nn.ReLU(),
            # 出力レイヤ
            nn.Linear(△△, num_classes)
        )

    def forward(self, x):
        return self.network(x)
 
    def predict(self, x):
        """予測クラスを返す"""
        with torch.no_grad():
            output = self.forward(x)
            _, predicted = torch.max(output, 1)
            return predicted

# モデルのインスタンス化
model = MultiLayerPerceptron(input_size=n_features, num_classes=n_classes).to(device)

# モデルの学習と予測
print("\n学習，始めるよ !!!")
train_losses, test_accuracies, train_accuracies = train_model(
    model, X_train, y_train, X_test, y_test,
    epochs=200, batch_size=32
)

print("\n最終結果:")
print(f"テストデータに対する正解率: {test_accuracies[-1]:.4f}")

### 2.3. ネットワーク構造の最適化

- 以下のハイパーパラメータを変更しながら，複数の MLPモデルを訓練し，性能を比較してください。
    - hidden_layer_sizes（ (50,), (100,), (50, 50), (100, 50), (100, 100)）
    - activation（'relu', 'tanh'）
    - solver（'adam', 'sgd'）

ハイパーパラメータの組み合わせは全部で20通り（$5 \times 2 \times 2$）あります。すべての組み合わせを試したい場合は、グリッドサーチを使用します。

In [ ]:
# 新しいMLPモデルの定義はここに追加してください
class MLP(nn.Module):

    def __init__(self, input_size, num_classes, hidden_layer_sizes=(10,), activation='relu'):
        torch.manual_seed(42)
        super(MLP, self).__init__()

        # 活性化関数のリスト
        # ヒント：nn.ReLU()、nn.Tanh()
        activate_functions = {
            'relu': 〇〇〇,
            'tanh': 〇〇〇,
        }

        # ネットワークの各層を格納するModuleListを初期化
        layers = nn.ModuleList()

        # 各層のサイズを決定するリストを作成
        layer_sizes = [input_size] + list(hidden_layer_sizes) + [num_classes]

        # 隣接する層間の線形変換(nn.Linear)と活性化関数(nn.ReLU)を順次追加
        for i in range(len(layer_sizes) - 1):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            # 出力層（最後の層）以外は活性化関数を追加
            if i < len(layer_sizes) - 2:
                layers.append(activate_functions[activation])

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

    def predict(self, x):
        """Returns predicted class labels"""
        with torch.no_grad():
            output = self.forward(x)
            _, predicted = torch.max(output, 1)
            return predicted

In [ ]:
# ハイパーパラメータの設定
hidden_layer_sizes_list = [〇〇〇〇〇〇〇〇]
activations_list = [〇〇〇〇〇〇〇〇]
solvers_list = [〇〇〇〇〇〇〇〇]

# 最適なハイパーパラメータの設定および
best_accuracy = 0
best_params = None
best_model = None

# すべてのハイパーパラメータの組合せを試す
for hidden_layer_sizes in hidden_layer_sizes_list:
    for activation in activations_list:
        for solver in solvers_list:
            print(f"ハイパーパラメータの設定: hidden_layer_sizes={hidden_layer_sizes}, activation={activation}, solver={solver}")
            model = MLP(
                input_size=2,
                num_classes=10,
                hidden_layer_sizes=hidden_layer_sizes,  # MultiLayerPerceptronクラスを変更してhidden_layer_sizeを引数として受け取るようにする
                activation=activation  # MultiLayerPerceptronクラスを変更してactivationを引数として受け取るようにする
            ).to(device)
            train_loss_history, train_acc_history, test_acc_history = train_model(
                model,
                X_train, y_train,
                X_test, y_test,
                epochs=500,
                solver=solver  # train_perceptron関数を変更してsolverを引数として受け取るようにする
            )
            # もし学習したモデルが前のモデルより高い正解率を得た場合、ハイパーパラメータの設定および正解率を保存する
            if test_acc_history[-1] > best_accuracy:
                best_accuracy = test_acc_history[-1]
                best_params = (hidden_layer_sizes, activation, solver)
                best_model = model

print(f"最適なハイパーパラメータの設定: hidden_layer_sizes={best_params[0]}, activation={best_params[1]}, solver={best_params[2]} \n正解率={best_accuracy}")

### 2.4. 最適なネットワーク構造について考察してください。